In [ ]:
import numpy as np

dataset = np.load("ZH_radar_dataset.npy") # dataset.shape == (288, 14, 360, 240)

# Compute the maximum reflectivity value across the scan axis (axis=1)
compressed_dataset = np.max(dataset, axis=1)  # shape: (288, 360, 240)

print("Compressed dataset shape:", compressed_dataset.shape)

# Optional: Save it
np.save("Data/ZH_max_across_scans.npy", compressed_dataset)
print("Saved compressed dataset to ZH_max_across_scans.npy")

## Movie of reflectivity progression through time

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.animation as animation
# from IPython.display import display, HTML
# from scipy.ndimage import binary_dilation
# from skimage.measure import find_contours
# from matplotlib.path import Path

# plt.rcParams['animation.embed_limit'] = 100
# data = np.load("Data/ZH_max_across_scans.npy")
# STORM_THRESHOLD = 45
# AREA_THRESHOLD = 15  # minimum number of pixels to qualify as a storm

# fig, ax = plt.subplots(figsize=(6, 7))
# cmap = plt.get_cmap("jet")

# img = ax.imshow(data[0], cmap=cmap, vmin=0, vmax=80)
# plt.colorbar(img, ax=ax, label="Reflectivity (dBZ)")
# title = ax.set_title("Radar Reflectivity - Time Step 0")

# storm_lines = []

# def update(frame):
#     global storm_lines
#     frame_data = data[frame]
#     img.set_data(frame_data)
#     title.set_text(f"Radar Reflectivity - Time Step {frame}")

#     # Remove previous lines
#     for line in storm_lines:
#         line.remove()
#     storm_lines = []

#     # Threshold + dilation
#     mask = frame_data > STORM_THRESHOLD
#     dilated_mask = binary_dilation(mask, iterations=3)

#     # Find storm region contours
#     contours = find_contours(dilated_mask.astype(float), 0.5)

#     for contour in contours:
#         path = Path(contour[:, ::-1])  # x, y
#         # Create a grid to count pixels inside the contour
#         x_grid, y_grid = np.meshgrid(np.arange(frame_data.shape[1]), np.arange(frame_data.shape[0]))
#         coords = np.vstack((x_grid.ravel(), y_grid.ravel())).T
#         inside = path.contains_points(coords).reshape(frame_data.shape)

#         area = np.sum(mask & inside)  # count original high-reflectivity pixels inside this contour

#         if area >= AREA_THRESHOLD:
#             line, = ax.plot(contour[:, 1], contour[:, 0], color='red', linewidth=2)
#             storm_lines.append(line)

#     return [img] + storm_lines

# plt.close(fig)
# ani = animation.FuncAnimation(fig, update, frames=data.shape[0], interval=100)
# display(HTML(ani.to_jshtml()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, HTML
from scipy.ndimage import binary_dilation
from skimage.measure import find_contours
from matplotlib.path import Path

def animate_storms(data, storm_threshold=45, area_threshold=15, dilation_iterations=5, interval=100):
    """
    Animate storm detection over time from radar reflectivity data.

    Parameters:
    - data: np.ndarray of shape (T, H, W)
    - storm_threshold: float, reflectivity threshold (dBZ) to identify storms
    - area_threshold: int, minimum number of pixels for a region to be considered a storm
    - dilation_iterations: int, number of binary dilation iterations to smooth/merge storms
    - interval: int, time between frames in ms

    Returns:
    - ani: matplotlib.animation.FuncAnimation object
    """
    fig, ax = plt.subplots(figsize=(6, 7))
    cmap = plt.get_cmap("jet")

    img = ax.imshow(data[0], cmap=cmap, vmin=0, vmax=80)
    plt.colorbar(img, ax=ax, label="Reflectivity (dBZ)")
    title = ax.set_title("Radar Reflectivity - Time Step 0")

    storm_lines = []

    def update(frame):
        nonlocal storm_lines
        frame_data = data[frame]
        img.set_data(frame_data)
        title.set_text(f"Radar Reflectivity - Time Step {frame}")

        # Remove previous storm lines
        for line in storm_lines:
            line.remove()
        storm_lines = []

        # Apply threshold and dilation
        mask = frame_data > storm_threshold
        dilated_mask = binary_dilation(mask, iterations=dilation_iterations)

        # Extract contours from binary mask
        contours = find_contours(dilated_mask.astype(float), 0.5)

        for contour in contours:
            path = Path(contour[:, ::-1])  # x, y
            xg, yg = np.meshgrid(np.arange(frame_data.shape[1]), np.arange(frame_data.shape[0]))
            coords = np.vstack((xg.ravel(), yg.ravel())).T
            inside = path.contains_points(coords).reshape(frame_data.shape)

            area = np.sum(mask & inside)

            if area >= area_threshold:
                line, = ax.plot(contour[:, 1], contour[:, 0], color='red', linewidth=2)
                storm_lines.append(line)

        return [img] + storm_lines

    plt.close(fig)  # Prevent duplicate static plot
    ani = animation.FuncAnimation(fig, update, frames=data.shape[0], interval=interval)
    return ani

In [ ]:
data = np.load("Data/ZH_max_across_scans.npy")  # Your 3D radar data

ani = animate_storms(data)
display(HTML(ani.to_jshtml()))

In [ ]:
# split the data into training and testing sets
# We are actually just interested in storm formation in our testing set. Because this is where we will evalueate if our model is able to predict storm formation.
train_size = int(len(compressed_dataset) * 0.6)
train_data = data[:train_size]
test_data = data[train_size:]

In [ ]:
ani = animate_storms(test_data)
display(HTML(ani.to_jshtml()))

## Algorithm that detects the presences of storms (as can be seen in the animation above). Also counts number of storms

In [ ]:
def detect_storms(data, reflectivity_threshold=45, area_threshold=15, dilation_iterations=5):
    """
    Detects storms in radar data and returns their count and coordinates at each time step.

    Parameters:
    - data: ndarray of shape (T, H, W)
    - reflectivity_threshold: threshold for storm detection
    - area_threshold: minimum storm area
    - dilation_iterations: dilation iterations

    Returns:
    - List of dictionaries containing storm count and coordinates per frame.
    """
    results = []

    for t in range(data.shape[0]):
        frame_data = data[t]
        mask = frame_data > reflectivity_threshold
        dilated_mask = binary_dilation(mask, iterations=dilation_iterations)
        contours = find_contours(dilated_mask.astype(float), 0.5)

        storms_in_frame = []

        for contour in contours:
            path = Path(contour[:, ::-1])  # (x, y) ordering

            # Grid coordinates
            x_grid, y_grid = np.meshgrid(
                np.arange(frame_data.shape[1]), np.arange(frame_data.shape[0])
            )
            coords = np.vstack((x_grid.ravel(), y_grid.ravel())).T
            inside = path.contains_points(coords).reshape(frame_data.shape)

            area = np.sum(mask & inside)

            if area >= area_threshold:
                storm_coords = contour[:, [1, 0]].tolist()  # (x, y) points
                storms_in_frame.append({"mask": inside.astype(int), "contour": storm_coords})

        frame_result = {
            "time_step": t,
            "storm_count": len(storms_in_frame),
            "storm_coordinates": [s["contour"] for s in storms_in_frame],
            "storm_masks": [s["mask"] for s in storms_in_frame],
        }

        results.append(frame_result)

    return results

In [ ]:
storm_data = detect_storms(test_data)
storm_data

In [ ]:
storm_data[2]  # Example of a time step

In [ ]:
# this gives the same result as the animation
import matplotlib.pyplot as plt
import numpy as np

# Choose a time step to visualize
frame_id = 10  # change to any frame you want
frame = test_data[frame_id]  # your original radar data

# Plot reflectivity
plt.figure(figsize=(6, 7))
plt.imshow(frame, cmap="jet", vmin=0, vmax=80)
plt.colorbar(label="Reflectivity (dBZ)")
plt.title(f"Storm Contours at Time Step {frame_id}")

# Plot storm contours from precomputed storm_data
for storm in storm_data[frame_id]['storm_coordinates']:
    storm = np.array(storm)
    plt.plot(storm[:, 0], storm[:, 1], color='red', linewidth=2)

plt.tight_layout()
plt.show()

## Algorithm that detects the formation of NEW storms (and where they occur)

In [ ]:
def detect_new_storm_formations(data, reflectivity_threshold=45, area_threshold=15, dilation_iterations=5, overlap_threshold=0.1):
    """
    Detects new storm formations: frames, count of newly formed storms, and their coordinates.

    Returns:
    - List of dictionaries with 'time_step', 'new_storm_count', and 'new_storm_coordinates'.
    """
    storm_results = detect_storms(data, reflectivity_threshold, area_threshold, dilation_iterations)
    new_storms_summary = []

    previous_masks = []

    for frame_result in storm_results:
        new_count = 0
        new_coords = []
        current_masks = frame_result['storm_masks']
        current_coords = frame_result['storm_coordinates']

        for i, current in enumerate(current_masks):
            is_new = True
            for prev in previous_masks:
                overlap = np.sum(current & prev) / np.sum(current)
                if overlap > overlap_threshold:
                    is_new = False
                    break
            if is_new:
                new_count += 1
                new_coords.append(current_coords[i])

        new_storms_summary.append({
            "time_step": frame_result['time_step'],
            "new_storm_count": new_count,
            "new_storm_coordinates": new_coords
        })

        previous_masks = current_masks

    return new_storms_summary


In [ ]:
new_storms = detect_new_storm_formations(test_data)
new_storms

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# def plot_new_storms(data, new_storms_result, frame_id):
#     """
#     Plot reflectivity with newly formed storms for a given frame.
    
#     Parameters:
#     - data: 3D ndarray (T, H, W)
#     - new_storms_result: output of detect_new_storm_formations
#     - frame_id: integer time step
#     """
#     # Get the frame and new storm data
#     frame = data[frame_id]
#     frame_entry = next((f for f in new_storms_result if f["time_step"] == frame_id), None)

#     if not frame_entry or frame_entry["new_storm_count"] == 0:
#         print(f"No new storms detected at time step {frame_id}.")
#         return

#     plt.figure(figsize=(6, 7))
#     plt.imshow(frame, cmap="jet", vmin=0, vmax=80)
#     plt.colorbar(label="Reflectivity (dBZ)")
#     plt.title(f"New Storms at Time Step {frame_id}")

#     for contour in frame_entry["new_storm_coordinates"]:
#         contour = np.array(contour)
#         plt.plot(contour[:, 0], contour[:, 1], color='lime', linewidth=2, label="New Storm")

#     plt.legend(loc="upper right")
#     plt.tight_layout()
#     plt.show()

# # Example usage
# new_storms = detect_new_storm_formations(test_data)
# # Plot for time step X(frame_id)
# plot_new_storms(test_data, new_storms, frame_id=7)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

def animate_new_storms(data, new_storms_result):
    fig, ax = plt.subplots(figsize=(6, 7))
    cmap = plt.get_cmap("jet")

    img = ax.imshow(data[0], cmap=cmap, vmin=0, vmax=80)
    title = ax.set_title("New Storms at Time Step 0")
    colorbar = plt.colorbar(img, ax=ax, label="Reflectivity (dBZ)")
    storm_lines = []

    def update(frame_id):
        nonlocal storm_lines
        frame = data[frame_id]
        img.set_data(frame)
        title.set_text(f"New Storms at Time Step {frame_id}")

        # Remove previous storm outlines
        for line in storm_lines:
            line.remove()
        storm_lines = []

        # Check if there are new storms at this frame
        frame_entry = next((f for f in new_storms_result if f["time_step"] == frame_id), None)
        if frame_entry and frame_entry["new_storm_count"] > 0:
            for contour in frame_entry["new_storm_coordinates"]:
                contour = np.array(contour)
                line, = ax.plot(contour[:, 0], contour[:, 1], color='lime', linewidth=2)
                storm_lines.append(line)

        return [img, title] + storm_lines
    plt.close(fig)
    ani = animation.FuncAnimation(fig, update, frames=data.shape[0], interval=200)
    return ani

In [ ]:
new_storms = detect_new_storm_formations(test_data)
ani = animate_new_storms(test_data, new_storms)
display(HTML(ani.to_jshtml()))

## Comparing predicted and actual storm formations

In [ ]:
def weighted_mae_high_reflectivity(threshold=20.0, zh_max=75.0, high_weight=5.0):
    """Returns a custom MAE loss that upweights pixels where ZH > threshold (in dBZ)."""
    norm_thresh = threshold / zh_max

    def loss(y_true, y_pred):
        # Weight = high_weight if y_true > threshold else 1.0
        weights = tf.where(y_true > norm_thresh, high_weight, 1.0)

        abs_error = tf.abs(y_true - y_pred)
        weighted_error = abs_error * weights

        return tf.reduce_mean(weighted_error)

    return loss

In [ ]:
from tensorflow.keras.models import load_model
model = load_model("checkpoints/normalized_convlstm_model.keras", custom_objects={"loss": weighted_mae_high_reflectivity})

In [ ]:
import numpy as np
data = np.load("Data/ZH_radar_dataset.npy")  # shape: (288, 14, 360, 240)
data = np.transpose(data, (0, 2, 3, 1))  # → (288, 360, 240, 14)
data[data < 0] = 0
ZH_MAX = np.max(data)
data_norm = data / ZH_MAX  # keep 0s as-is

SEQ_LEN = 10
X, y = [], []
for i in range(len(data_norm) - SEQ_LEN):
    X.append(data_norm[i:i+SEQ_LEN])   # shape: (10, 360, 240, 14)
    y.append(data_norm[i+SEQ_LEN])     # shape: (360, 240, 14)
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

split = int(len(X) * 0.6)
#X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

In [ ]:
y_test = y_test*ZH_MAX 
y_test

In [ ]:
pred = model.predict(X_test) *ZH_MAX 

In [ ]:
pred.shape

In [ ]:
pred_max_cappi = np.max(pred, axis=-1)
true_max_cappi = np.max(y_test, axis=-1)

In [ ]:
new_storms_pred = detect_new_storm_formations(pred_max_cappi)
new_storms_true = detect_new_storm_formations(true_max_cappi)

In [ ]:
ani1 = animate_new_storms(pred_max_cappi, new_storms_pred)
ani2 = animate_new_storms(true_max_cappi, new_storms_true)

In [ ]:
from IPython.display import display, HTML

html = f"""
<div style="display: flex; gap: 20px;">
  <div>{ani1.to_jshtml()}</div>
  <div>{ani2.to_jshtml()}</div>
</div>
"""

display(HTML(html))

Create some sort of comparison between the true and predicted storms. So if the region of the new storm overlap let's say 70%, then we classify it as a correct prediction. And also keep track of the correct predictions, the missed predictions and the false predictions. 